# Clustering of all data

Read in all necessary packages:

In [55]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))
%matplotlib inline

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pandas.io.parsers import read_csv
import scanpy as sc
import numpy as np
from functools import reduce
from anndata import AnnData, read_h5ad
import loompy
import collections
import scipy
import math
from matplotlib.colors import LinearSegmentedColormap
import random
from sklearn.mixture import GaussianMixture
from scipy.stats import poisson

import scipy
import tqdm
from statsmodels.stats.multitest import multipletests
import matplotlib.patches as mpatches

/var/folders/pb/hvc2shb53x194_v0c3h_jyccsf3dhb/T/ipykernel_35093/284414565.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [56]:
sc.set_figure_params(dpi = 80)
sns.set_style("white")
sns.set_context("talk")
sns.set_style("ticks")

In [57]:
greens_r = sns.blend_palette(['black','#238b45', '#75c477', '#c8e9c1' ], as_cmap = True)
purples_r = sns.blend_palette(['black','#6a51a3', '#9e9bc8', '#dadaeb'], as_cmap = True)
oranges_r = sns.blend_palette(['black','#d94801', '#fd8e3d', '#fdd1a3'], as_cmap = True)

k27_k4 = sns.blend_palette(['#6a51a3', '#9e9bc8', '#dadaeb','#c8e9c1','#75c477','#238b45'], as_cmap = True)
k4_k27 = sns.blend_palette(['#238b45', '#75c477', '#c8e9c1' ,'#dadaeb','#9e9bc8','#6a51a3' ], as_cmap = True)
Green='#238b45'
Purple='#6a51a3'
Orange='#d94801'


In [58]:
cols = ['grey','skyblue','orange','red']
cols2 = ['white','deepskyblue','orange','red','grey']

leg_dict2 = {'negative': 'grey','positive': 'skyblue','going up': 'orange','going down': 'red'}
coldict=dict(zip(list(pd.DataFrame([0,1,2,3,4]).astype('category')[0]), cols2))
leg_dict = {'negative': 'white', 'positive': 'deepskyblue', 'going up': 'orange', 'going down': 'red'}

## formulas

In [5]:
pseudo = []
def bintable(df, stepsize):    
    df = df.sort_values(by = ['hour','pseudotime'],ascending=True)
    df['order'] = list(range(0,len(df)))
    df['steps'] = df['order'].apply(lambda x: np.floor(x/stepsize))
    binned_table_mean = {}
    binned_table_sum = {}
    timepoint = []
    for step in df['steps'].unique():
        binned_table_mean[step] = df[df['steps'] == step].mean(numeric_only=True)
        binned_table_sum[step] = df[df['steps'] == step].sum(numeric_only=True)
        timepoint.append(df[df['steps'] == step]['hour'].value_counts().sort_values(ascending=False).index[0])
    binned_pseudo = pd.DataFrame(pd.DataFrame(binned_table_mean).T['pseudotime'])
    binned_pseudo['timepoint'] = timepoint
    binned_final_mean = pd.DataFrame(binned_table_mean).T.drop(['order','steps','pseudotime'],axis = 1)
    binned_final_sum = pd.DataFrame(binned_table_sum).T.drop(['order','steps','pseudotime'],axis = 1)
    binned_pseudo = binned_pseudo.iloc[:-1,:]
    binned_final_mean = binned_final_mean.iloc[:-1,:]
    binned_final_sum = binned_final_sum.iloc[:-1,:]
    return binned_pseudo, binned_final_mean, binned_final_sum

In [6]:
def gene_ttest(b_df, mean = 2.5, fold = 1.5, pv = 2.3, cutoff = 10):
    start = np.log2((b_df.iloc[0:cutoff])+1)
    stop = np.log2((b_df.iloc[-cutoff:])+1)
    testout = pd.Series(index=start.columns, dtype='float')
    for gene in start.columns:
        s , p = stats.ttest_ind(start[gene],stop[gene])
        testout[gene] = p
    testresults = pd.DataFrame(columns=['fold','p','state','mean'], index = start.columns)
    testresults['fold'] = (stop.mean(axis=0)-start.mean(axis=0))
    testresults['mean'] = (start.mean(axis=0)+stop.mean(axis=0))/2
    testresults['p'] = (-np.log10(testout)).fillna(1)
    testresults['state'] = 0
    testresults.loc[((testresults['mean'] > mean)),'state'] = 1
    testresults.loc[((abs(testresults['fold']) > fold)&(testresults['p'] > pv)&(testresults['mean'] > mean)),'state'] = 2
    
    for gene in list(testresults[(testresults['fold']  < 0 )  & (testresults['state']  == 2)].index):
        testresults.loc[gene, 'state'] = 3   
    
    return testresults


In [177]:
def anova_posthoc(b_df,layer, fold = 1.5, pv = 2.3, mean = 2.5, post_hoc = False, states = False): #k4 mean 1.2, k27 mean 4.5, rna mean 2.5
    results_out = collections.defaultdict(lambda: collections.defaultdict(collections.Counter ))
    statistic = []
    pval_tukey = []

    print('performing statistics...')
    for gene in b_df['gene'].unique():
        if gene == b_df['gene'].unique()[int(len(b_df['gene'].unique())*0.25)]:
            print('25% done!')
        if gene == b_df['gene'].unique()[int(len(b_df['gene'].unique())*0.5)]:
            print('50% done!')  
        if gene == b_df['gene'].unique()[int(len(b_df['gene'].unique())*0.75)]:
            print('75% done!')
            
        #perform anova per gene
        s, p = stats.f_oneway(*b_df[b_df['gene']==gene].pivot(columns='step', index='rep',values=layer).T.values.tolist()) 
        
        if post_hoc == True:
            pvals_tukey = []
            statistics_tukey = []
            if p < 0.05: # when anova is significant
                # perform tukey post-hoc test
                tukey = stats.tukey_hsd(*b_df[b_df['gene']==gene].pivot(columns='step', index='rep',values=layer).T.values.tolist()) 
                pvals_tukey = tukey.pvalue[range(0, b_df['step'].max()),range(1,b_df['step'].max()+1)]
                statistics_tukey = tukey.statistic[range(0, b_df['step'].max()),range(1,b_df['step'].max()+1)]
                significant_step = (pvals_tukey<0.05).sum()
                save_diff = min(range(len(pvals_tukey)), key=pvals_tukey.__getitem__)
                step_tukey = str(range(0,b_df['step'].max())[save_diff])
                pval_tukey = pvals_tukey[save_diff]
                statistic_tukey = statistics_tukey[save_diff]
            else: # when anova is not significant
                pvals_tukey.append([0] * b_df['step'].max())
                statistics_tukey.append([0] * b_df['step'].max())
                significant_step = 0
                step_tukey = 0
                pval_tukey = 1
                statistic_tukey = 0 
                
            # save tukey results
            results_out[gene]['tukey_step'] = float(step_tukey)
            results_out[gene]['significant_steps'] = float(significant_step)
            results_out[gene]['tukey_pval'] = np.float64(pval_tukey)
            results_out[gene]['tukey_statistic'] = float(statistic_tukey)
            
            results_out[gene]['tukey_pval_all'] = str(pvals_tukey)
            results_out[gene]['tukey_statistic_all'] = str(statistics_tukey)
            
        # save anova results
        results_out[gene]['p_raw'] = np.float64(p)
        results_out[gene]['p'] = (-np.log10(np.float64(p)))
        all_means = b_df[b_df['gene']==gene].pivot(columns='step', index='rep',values=layer).mean(axis=0)
        results_out[gene]['mean'] = all_means.mean()
        results_out[gene]['fold'] = (max(all_means) - min(all_means))
        
    # make final df, make p-values float
    print('saving dataframe...')
    testresults = pd.DataFrame(results_out).T
    testresults['tukey_pval'] = testresults['tukey_pval'].astype('float64')
    testresults['p_raw'] = testresults['p_raw'].astype('float64')
    
    if states == True:
        print('determining states...')
        testresults['state'] = 0
        testresults.loc[((testresults['mean'] > mean)),'state'] = 1
        testresults.loc[((abs(testresults['fold']) > fold)&(testresults['p'] > pv)&(testresults['mean'] > mean)),'state'] = 2

        for gene in list(testresults[(testresults['fold']  < 0 )  & (testresults['state']  == 2)].index):
            testresults.loc[gene, 'state'] = 3   

    print('all done!')
    return testresults



## K4me3 TSS tables

In [8]:
table1 = pd.read_csv('../data/peter/20240119_Fig5_ECT_PB_counts.tsv',
                            index_col = 0, header = 0,low_memory = False, sep = '\t')





In [9]:
table2 = pd.read_csv('../data/peter/20240119_Fig5_ECT_PB_sub_counts.tsv',
                            index_col = 0, header = 0,low_memory = False, sep = '\t')





In [10]:
table3 = pd.read_csv('../data/peter/20240119_Fig5_ECT_PB_meta.tsv',
                            index_col = 0, header = 0,low_memory = False, sep = '\t')





In [11]:
table1

,gene,step,RNA,K4,K27,lin
0,Xkr4,0,98.0,31.0,227.0,Ectoderm
1,Rp1,0,99.0,2.0,220.0,Ectoderm
2,Sox17,0,4.0,86.0,318.0,Ectoderm
3,Mrpl15,0,981.0,116.0,46.0,Ectoderm
4,Rgs20,0,620.0,106.0,186.0,Ectoderm
...,...,...,...,...,...,...
443745,Arhgap6,24,192.0,6.0,86.0,Ectoderm
443746,Mid1,24,675.0,15.0,3.0,Ectoderm
443747,Uty,24,394.0,6.0,1.0,Ectoderm
443748,Kdm5d,24,5.0,0.0,3.0,Ectoderm


In [12]:
table2

,gene,step,RNA,K4,K27,lin,rep
0,Xkr4,0,23.0,8.0,54.0,Ectoderm,1
1,Rp1,0,18.0,0.0,57.0,Ectoderm,1
2,Sox17,0,2.0,24.0,83.0,Ectoderm,1
3,Mrpl15,0,257.0,28.0,11.0,Ectoderm,1
4,Rgs20,0,127.0,23.0,46.0,Ectoderm,1
...,...,...,...,...,...,...,...
1774995,Arhgap6,24,27.0,1.0,21.0,Ectoderm,4
1774996,Mid1,24,183.0,7.0,0.0,Ectoderm,4
1774997,Uty,24,150.0,2.0,0.0,Ectoderm,4
1774998,Kdm5d,24,3.0,0.0,2.0,Ectoderm,4


In [13]:
table3

,step,PT,UMAP1,UMAP2,lin,RT
0,0,0.557643,11.386752,4.404837,Ectoderm,mESCs
1,1,1.209081,11.395923,4.444204,Ectoderm,mESCs
2,2,1.996789,11.507477,4.357666,Ectoderm,mESCs
3,3,3.028253,11.697371,4.320375,Ectoderm,mESCs
4,4,9.144476,10.309409,4.533459,Ectoderm,mESCs
5,5,25.753560,5.108832,5.900985,Ectoderm,Gastd3
6,6,26.544820,4.706913,6.206382,Ectoderm,Gastd3
7,7,26.976249,4.225320,6.187308,Ectoderm,Gastd3
8,8,27.629920,3.421360,5.960229,Ectoderm,Gastd4
9,9,28.351053,2.752185,5.710185,Ectoderm,Gastd4


In [15]:
table2['RNA_log2'] = np.log2(table2['RNA']+1)

In [16]:
from scipy import stats

In [21]:
table2['K4_log2'] = np.log2(table2['K4']+1)
table2['K27_log2'] = np.log2(table2['K27']+1)

In [178]:
table_sub2 = table2[table2['gene'].isin(list(table2['gene'][0:8].unique()))]

In [180]:
fast_test = anova_posthoc(table_sub2, layer = 'RNA_log2', post_hoc = True, states = True)

performing statistics...
25% done!
50% done!
75% done!
saving dataframe...
determining states...
done!


In [162]:
fast_test

,tukey_step,significant_steps,tukey_pval,tukey_statistic,tukey_pval_all,tukey_statistic_all,p_raw,p,mean,fold,state
Xkr4,4.0,5.0,1.221245e-15,-3.411016,[1.00000000e+00 1.00000000e+00 1.00000000e+00 ...,[-0.01117125 -0.06568588 0.01361195 -2.434438...,1.718792e-56,55.764777,7.740003,6.651464,2
Rp1,4.0,0.0,2.940270e-01,1.02524,[1. 1. 1. 1. 0...,[ 0.0885286 0.07318366 -0.00492904 -0.150221...,6.133692e-12,11.212278,3.516209,2.251238,2
Sox17,4.0,0.0,9.261700e-01,-1.25,[1. 0.99997318 1. 0.99999979 0...,[-0.40935748 0.70183873 0. -0.542481...,3.367114e-03,2.472742,1.751674,2.362803,0
Mrpl15,8.0,0.0,3.133450e-01,0.373719,[1. 0.99999282 1. 0.63728734 1...,[ 0.01382878 -0.13218491 -0.04345539 0.316686...,3.369768e-41,40.4724,7.151692,2.35522,2
Rgs20,4.0,2.0,1.599967e-02,0.875411,[1. 1. 0.9999862 0.5819938 0...,[ 0.046975 -0.03006439 -0.23358333 0.555637...,3.621453e-18,17.441117,6.728113,1.930895,2


In [132]:
test.head(3)

,fold,p,state,mean,p_raw,tukey_step,significant_steps,tukey_pval,tukey_statistic,tukey_pval_all,tukey_statistic_all
Xkr4,6.651464,55.764777,1,7.740003,1.718792e-56,4,5.0,1.221245e-15,-3.411016,"[1.0, 1.0, 1.0, 1.1801670751765414e-13, 1.2212...","[-0.011171253654804758, -0.06568587764325695, ..."
Rp1,2.251238,11.212278,1,3.516209,6.133692e-12,4,0.0,2.940270e-01,1.025240,"[1.0, 1.0, 1.0, 0.9999999999999284, 0.29402701...","[0.08852860311524502, 0.0731836600123037, -0.0..."
Sox17,2.362803,2.472742,1,1.751674,3.367114e-03,4,0.0,9.261700e-01,-1.250000,"[0.9999999994008426, 0.9999731830887576, 1.0, ...","[-0.4093574801538229, 0.7018387305144009, 0.0,..."


In [105]:
fast_test['p_raw'].astype('float64')

Xkr4     1.718792e-56
Rp1      6.133692e-12
Sox17    3.367114e-03
Name: p_raw, dtype: float64

In [ ]:
fig,ax = plt.subplots(2,3,figsize = (9,6)) 
                                #gridspec_kw={'hspace': 0.1, 'height_ratios':[2,2,2,2]})
      
ax = ax.flatten()
sns.histplot(test3['p'], color = 'mediumseagreen',ax=ax[0])
sns.histplot(test3['fold'], color = 'cornflowerblue',ax=ax[1])
sns.histplot(test3['mean'], color = 'purple',ax=ax[2])

sns.histplot(test3['tukey_step'].astype(int), color = 'goldenrod',ax=ax[3])
sns.histplot(test3['significant_steps'], color = 'coral',ax=ax[4])
sns.histplot(test3['tukey_statistic'],color = 'crimson',ax=ax[5])

ax[1].set_ylabel('')
ax[2].set_ylabel('')
ax[4].set_ylabel('')
ax[5].set_ylabel('')

ax[0].set_title('pval')
ax[1].set_title('fold')
ax[2].set_title('mean')
#plt.show()